In [0]:
# ============================================================
# PAYCE DATA PLATFORM — Silver Layer Configuration
# Notebook:    02_silver_processing
# Layer:       Silver
# Description: Reusable config — run this cell first each session
# ============================================================

# S3 base paths
S3_RAW_BASE    = "s3://payce-data-platform/raw"
S3_BRONZE_BASE = "s3://payce-data-platform/bronze"
S3_SILVER_BASE = "s3://payce-data-platform/silver"

# Catalog preferences
CATALOG = "payce"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"

print("Payce Silver config loaded")
print(f" CATALOG: {CATALOG}")
print(f" BRONZE:  {BRONZE}")
print(f" SILVER:  {SILVER}")

In [0]:
# Check existing catalogs
spark.sql("SHOW CATALOGS").show()

In [0]:
# CREATE THE PAYCE CATALOG
spark.sql("CREATE CATALOG IF NOT EXISTS payce")

# CREATE the medallion schema
spark.sql("CREATE SCHEMA IF NOT EXISTS payce.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS payce.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS payce.gold")

# CONFIRM
spark.sql("SHOW SCHEMAS IN payce").show()

In [0]:
# ============================================================
# Register existing Bronze Delta files as Unity Catalog tables
# ============================================================

S3_BRONZE_BASE = "s3://payce-data-platform/bronze"

# List of datasets to register
datasets = ["paysim", "ieee_cis", "home_credit", "lending_club"]

for ds in datasets:
    spark.sql(f"""
              CREATE TABLE IF NOT EXISTS payce.bronze.{ds}
              USING DELTA
              LOCATION '{S3_BRONZE_BASE}/{ds}/'
              """)
    print(f" Registed payce.bronze.{ds}")

# Confirm all tables
spark.sql("SHOW TABLES IN payce.bronze").show()

In [0]:
# Test querying via catalog
spark.sql("SELECT * from payce.bronze.paysim LIMIT 5").show()

In [0]:
# ===============================================================
# SILVER - PaySim Exploration
# Understand the data before transforming
# ===============================================================

# Load Bronze PaySim
df = spark.table(f"{BRONZE}.paysim")

# 1. Look at the schema (column names + data types)
print("SCHEMA:")
df.printSchema()

# 2. Row Count
print(f"\n Total rows: {df.count():,}")

# 3. Preview a few rows
df.show(5)

In [0]:
# Check transaction types
print("TRANSACTION TYPES:")
df.groupBy("type").count().show()

# Check if _rescued_data has any bad records
print("RESCUED (bad) records:")
df.filter("_rescued_data IS NOT NULL").count()

In [0]:
from pyspark.sql.functions import col, when

# Load Bronze
df = spark.table(f"{BRONZE}.paysim")

# Cast columns to correct data types
df_silver = (
    df
    .withColumn("step",             col("step").cast("int"))
    .withColumn("amount",           col("amount").cast("double"))
    .withColumn("oldbalanceOrg",    col("oldbalanceOrg").cast("double"))
    .withColumn("newbalanceOrig",   col("newbalanceOrig").cast("double"))
    .withColumn("oldbalanceDest",   col("oldbalanceDest").cast("double"))
    .withColumn("newbalanceDest",   col("newbalanceDest").cast("double"))
    .withColumn("isFraud",          col("isFraud").cast("int"))
    .withColumn("isFlaggedFraud",   col("isFlaggedFraud").cast("int"))
)

# Drop the _rescued_data column (confirmed as empty)
df_silver = df_silver.drop("_rescued_data")


# Remove the duplicates rows
df_silver = df_silver.dropDuplicates()

# Add a derived column - balance change for the origin account 
df_silver = df_silver.withColumn(
    "balance_change_orig",      col("oldbalanceOrg") - col("newbalanceOrig")
)


# Write to silver as a managed delta table 

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.paysim")
)

print("payce.silver.paysim created successfully")

In [0]:
%sql
select count(*) from payce.silver.paysim

In [0]:
df_silver.printSchema()

In [0]:
# ============================================================
# SILVER — IEEE-CIS Exploration
# ============================================================

# Load Bronze IEEE-CIS
df_ieee = spark.table(f"{BRONZE}.ieee_cis")

# 1. Schema
print("SCHEMA: ")
df_ieee.printSchema()

# 2. Row Count
print("f\n Total rows: {df_ieee.count():,}")

# 3. How many columns
print(f"Total Columns: {len(df_ieee.columns)}")

In [0]:
# Check Fraud Distribution 
df_ieee.groupBy("isFraud").count().show()

In [0]:
### isFraud contains indicators 1,0, NULL, 0.5. We need to remove and exclude NULLS and 0.5 from this data

In [0]:
from pyspark.sql.functions import col 

# Keep only labelled training rows
df_ieee_clean = df_ieee.filter(col("isFraud").isin("0","1"))

# Confirm the fix
df_ieee_clean.groupBy("isFraud").count().show()

In [0]:
# ============================================================
# SILVER — IEEE-CIS Transformation
# Source: payce.bronze.ieee_cis
# Target: payce.silver.ieee_cis
# Approach: Filter to labelled rows, curate ~15 BNPL-relevant
#           columns from the original 473
# ============================================================

from pyspark.sql.functions import col

# Load Bronze 
df_ieee = spark.table(f"{BRONZE}.ieee_cis")

# Keep only real and fraud indicators
df_ieee = df_ieee.filter(col("isFraud").isin("0","1"))

# Select only BNPL relevant columns from 473 
selected_columns = [
    "TransactionID",        # unique transaction key
    "isFraud",              # fraud income
    "TransactionDT",        # time (seconds from reference)
    "TransactionAmt",       # transaction amount in USD
    "ProductCD",            # product code
    "card4",                # card network
    "card6",                # card type (credit/debit)
    "P_emaildomain",        # purchaser email domain
    "addr1",                # billing region
    "C1",                   # count signal
    "C2",                   # count signal
    "D1",                   # days since first transaction
    "M1",                   # match flag
    "M6",                   # match flag
]

df_ieee = df_ieee.select(*selected_columns)


# Cast columns to correct data types
df_ieee_silver = (
    df_ieee
    .withColumn("isFraud",                   col("isFraud").cast("int"))
    .withColumn("TransactionDT",             col("TransactionDT").cast("long"))
    .withColumn("TransactionAmt",            col("TransactionAmt").cast("double"))
    .withColumn("addr1",                     col("addr1").cast("double").cast("int"))
    .withColumn("C1",                        col("C1").cast("double"))
    .withColumn("C2",                        col("C2").cast("double"))
    .withColumn("D1",                        col("D1").cast("double"))
)

# Remove duplicates
df_ieee_silver = df_ieee_silver.dropDuplicates(["TransactionID"])

# Write Silver 
(
    df_ieee_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.ieee_cis")
)

print("payce.silver.ieee_cis created successfully")

In [0]:
%sql
select count(*) from payce.silver.ieee_cis 